### RAG pipeling - Data ingestion to vector DB pipeline

In [3]:
import os
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter


/opt/anaconda3/envs/myenv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
### Read all the pdf's inside the directory

def load_pdfs_with_metadata(directory_path):
    all_docs = []

    for file in os.listdir(directory_path):
        if file.endswith(".pdf"):
            file_path = os.path.join(directory_path, file)

            # Load PDF
            loader = PyPDFLoader(file_path)
            documents = loader.load()

            # Add metadata
            for doc in documents:
                doc.metadata.update({
                    "source_file": file,
                    "file_path": file_path,
                    "file_type": "pdf"
                })

            all_docs.extend(documents)

    return all_docs

In [5]:
all_pdf_documents=load_pdfs_with_metadata("./data/pdf")

In [6]:
print(all_pdf_documents)

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-10T10:40:32+00:00', 'author': '', 'keywords': '', 'moddate': '2026-04-10T10:40:32+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/pdf/sanskar_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sanskar_resume.pdf', 'file_path': './data/pdf/sanskar_resume.pdf', 'file_type': 'pdf'}, page_content='Sanskar Vishwakarma\n+91 93215 97049 — sanskarv2004@gmail.com — Linkedin — Github\nEducation\nBachelor of Engineering (Computer Engineering)Jun 2021 – Jun 2025\nThakur College of Engineering and Technology9.48 CGPA\nHigher Secondary Certificate (HSC - PCM)Jun 2021\nNirmala College of Science and Commerce94%\nTechnical Skills\nCore Concepts:Data Structures, OOP, DBMS, API Integration.\nProgramming Languages:Java, JavaScript, TypeSc

In [7]:
### Chunking the documents into text

def chunk_documents(documents, chunk_size=1000, chunk_overlap=200):
    
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    
    chunked_docs = splitter.split_documents(documents)

    print(f"Split {len(documents)} documents into {len(chunked_docs)} chunks")

    if chunked_docs:
        print(f"\nExample chunk")
        print(f"\nContent: {chunked_docs[0].page_content[:200]}")
        print(f"\nMetadata: {chunked_docs[0].metadata}")

    return chunked_docs

In [8]:
chunks=chunk_documents(all_pdf_documents)

Split 1 documents into 4 chunks

Example chunk

Content: Sanskar Vishwakarma
+91 93215 97049 — sanskarv2004@gmail.com — Linkedin — Github
Education
Bachelor of Engineering (Computer Engineering)Jun 2021 – Jun 2025
Thakur College of Engineering and Technolog

Metadata: {'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-10T10:40:32+00:00', 'author': '', 'keywords': '', 'moddate': '2026-04-10T10:40:32+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/pdf/sanskar_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sanskar_resume.pdf', 'file_path': './data/pdf/sanskar_resume.pdf', 'file_type': 'pdf'}


### Embedding and VectorDB

In [9]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List,Dict,Any,Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [10]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self,model_name="all-MiniLM-L6-v2"):
        """
        Intialise the embedding manager

        Args:
            model_name: HugginFace model for sentence embeddings
        """

        self.model_name=model_name
        self.model=None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""

        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model=SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimesion: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loadig model {self.model_name}:{e}")
            raise

    def generate_embeddings(self,texts:List[str])->np.ndarray:
        """
        generate embeddings for a list of texts

        Args:
            texts: List of text strings to embed

        Returns:
            numpy array of embeddings with shape (len(texts),embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"generating embeddings for {len(texts)} texts...")
        embeddings=self.model.encode(texts,show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        
        return embeddings


### initialise the embedding manager
embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8579.01it/s]


Model loaded successfully. Embedding dimesion: 384


### Vector Store

In [11]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector Store"""

    def __init__(self,collection_name:str="pdf_documents",persist_directory:str="./data/vector_store"):
        """
        Initialises the vector store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name=collection_name
        self.persist_directory=persist_directory
        self.client=None
        self.collection=None
        self._initialise_store()

    def _initialise_store(self):
        """Intialise ChromaDB client and collection"""
        try:
            # Creating ChromaDB persistent client
            os.makedirs(self.persist_directory,exist_ok=True)
            self.client=chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection=self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description":"PDF document embeddings for RAG","hnsw:space": "cosine"}
            )
            print(f"Vector Store initialised, Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initialising vector store: {e}")
            raise

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
            """
            Add documents and their embeddings to the vector store
            
            Args:
                documents: List of LangChain documents
                embeddings: Corresponding embeddings for the documents
            """
            if len(documents) != len(embeddings):
                raise ValueError("Number of documents must match number of embeddings")
            
            print(f"Adding {len(documents)} documents to vector store...")
            
            # Prepare data for ChromaDB
            ids = []
            metadatas = []
            documents_text = []
            embeddings_list = []
            
            for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
                # Generate unique ID
                doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
                ids.append(doc_id)
                
                # Prepare metadata
                metadata = dict(doc.metadata)
                metadata['doc_index'] = i
                metadata['content_length'] = len(doc.page_content)
                metadatas.append(metadata)
                
                # Document content
                documents_text.append(doc.page_content)
                
                # Embedding
                embeddings_list.append(embedding.tolist())
            
            # Add to collection
            try:
                self.collection.add(
                    ids=ids,
                    embeddings=embeddings_list,
                    metadatas=metadatas,
                    documents=documents_text
                )
                print(f"Successfully added {len(documents)} documents to vector store")
                print(f"Total documents in collection: {self.collection.count()}")
                
            except Exception as e:
                print(f"Error adding documents to vector store: {e}")
                raise


vectorStore=VectorStore()

vectorStore
        

Vector Store initialised, Collection: pdf_documents
Existing documents in collection: 20


In [12]:
chunks

[Document(metadata={'producer': 'pdfTeX-1.40.27', 'creator': 'LaTeX with hyperref', 'creationdate': '2026-04-10T10:40:32+00:00', 'author': '', 'keywords': '', 'moddate': '2026-04-10T10:40:32+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) kpathsea version 6.4.1', 'subject': '', 'title': '', 'trapped': '/False', 'source': './data/pdf/sanskar_resume.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1', 'source_file': 'sanskar_resume.pdf', 'file_path': './data/pdf/sanskar_resume.pdf', 'file_type': 'pdf'}, page_content='Sanskar Vishwakarma\n+91 93215 97049 — sanskarv2004@gmail.com — Linkedin — Github\nEducation\nBachelor of Engineering (Computer Engineering)Jun 2021 – Jun 2025\nThakur College of Engineering and Technology9.48 CGPA\nHigher Secondary Certificate (HSC - PCM)Jun 2021\nNirmala College of Science and Commerce94%\nTechnical Skills\nCore Concepts:Data Structures, OOP, DBMS, API Integration.\nProgramming Languages:Java, JavaScript, TypeSc

In [13]:
### convert the text to embeddings
texts=[doc.page_content for doc in chunks]
texts

### Generate the embeddings
embeddings=embedding_manager.generate_embeddings(texts)
embeddings

### Store the embeddings and chunks into vector store
vectorStore.add_documents(chunks,embeddings)



generating embeddings for 4 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.01it/s]

Generated embeddings with shape: (4, 384)
Adding 4 documents to vector store...
Successfully added 4 documents to vector store
Total documents in collection: 24


### RAG Retriever Pipeline from Vector Store

In [14]:
class RAGRetriever:
    """Handles query based retrieval from the vector Store"""

    def __init__(self,vectorStore:VectorStore, embedding_manager:EmbeddingManager):
        """
        Initialise the retriever

        Args:
            vectorStore: Vector Store containing the document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vectorStore=vectorStore
        self.embedding_manager=embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant documents for a query
        
        Args:
            query: The search query
            top_k: Number of top results to return
            score_threshold: Minimum similarity score threshold
            
        Returns:
            List of dictionaries containing retrieved documents and metadata
        """
        print(self.vectorStore.collection.count())
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")
        
        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]
        
        # Search in vector store
        try:
            results = self.vectorStore.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            print(results['distances'][0])
            
            # Process results
            retrieved_docs = []
            
            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]
                
                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    # similarity_score = 1 - distance

                    similarity_score = 1 / (1 + distance)
                    
                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance': distance,
                            'rank': i + 1
                        })
                
                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No documents found")
            
            return retrieved_docs
            
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

        

In [15]:
ragRetreiver=RAGRetriever(vectorStore,embedding_manager)

In [16]:
ragRetreiver

In [17]:
ragRetreiver.retrieve("who is Sanskar")

24
Retrieving documents for query: 'who is Sanskar'
Top K: 5, Score threshold: 0.0
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 27.77it/s]

Generated embeddings with shape: (1, 384)
[1.6243634223937988, 1.6243634223937988, 1.6243634223937988, 1.6243634223937988, 1.6243634223937988]
Retrieved 5 documents (after filtering)


[{'id': 'doc_a6628916_0',
  'content': 'Sanskar Vishwakarma\n+91 93215 97049 — sanskarv2004@gmail.com — Linkedin — Github\nEducation\nBachelor of Engineering (Computer Engineering)Jun 2021 – Jun 2025\nThakur College of Engineering and Technology9.48 CGPA\nHigher Secondary Certificate (HSC - PCM)Jun 2021\nNirmala College of Science and Commerce94%\nTechnical Skills\nCore Concepts:Data Structures, OOP, DBMS, API Integration.\nProgramming Languages:Java, JavaScript, TypeScript, SQL.\nLibraries and FrameworksReact.js, Next.js, Node.js, Express.js, Spring Boot, REST APIs ,Tailwind CSS.\nDatabases:MongoDB, Redis, MySQL.\nDevOps & Tools:Docker, GitHub, Postman, SoapUI, JIRA, VS Code, IntelliJ IDEA, MS Office.\nAI Tools:Cursor , Antigravity , Claude , Codex CLI, Gemini CLI, GitHub Copilot.\nProfessional Experience\nAssociate Consultant— OracleJun 2025 – Apr 2026\n•Enhanced Oracle Flexcube core banking modules forYES Bank, delivering regulatory and business-critical\nenhancements aligned with R

### Integrating vectordb Context pipeline with LLM output

In [18]:
### Simple RAG pipeline with GROQ LLM

from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.getenv("GROQ_API_KEY")

llm=ChatGroq(
    groq_api_key=api_key,
    model_name="llama-3.1-8b-instant",
    temperature=0.1,
    max_tokens=1024
)

def rag_simple(query,retriever,llm,top_k=3):
    # retrieve the content
    results=retriever.retrieve(query,top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."
    
    prompt=f"""You are a helpful AI assistant. Use the following context to answer the question accurately and concisely.

    Context:
    {context}

    Question: {query}

    Answer: Provide a clear and informative answer based on the context above. If the context doesn't contain enough information to answer the question, say so."""

    response=llm.invoke([prompt.format(context,query)])
    return response.content
        


In [19]:
answer=rag_simple("Tell me the professional experience of this person?",retriever=ragRetreiver,llm=llm)

24
Retrieving documents for query: 'Tell me the professional experience of this person?'
Top K: 3, Score threshold: 0.0
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 39.71it/s]


Generated embeddings with shape: (1, 384)
[1.585943341255188, 1.585943341255188, 1.585943341255188]
Retrieved 3 documents (after filtering)


In [20]:
print(answer)

Based on the provided context, the person has the following professional experience:

1. **Backend Development**: They have experience in developing backends for applications, specifically:
	* Implementing watch later, subscriptions, and user authentication functionalities.
	* Developing RESTful APIs for video management and user interactions (BookMySeat project).
	* Designing and developing the backend for a movie ticket booking application (BookMySeat project).
	* Building RESTful APIs for movie listings, shows, seat selection, and bookings (BookMySeat project).
	* Implementing seat availability tracking and end-to-end booking workflow (BookMySeat project).
2. **Technical Leadership**: They have experience as a Technical Head at TCET Shastra Coding Club, where they:
	* Designed and managed technical aspects for 14+ coding events and 4 technical seminars.
	* Contributed to a 20-member technical team managed under the core team & Training and Placement Cell.

Overall, the person has ex

### Enhanced RAG pipeline

In [22]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt.format(context=context, query=query)])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("What is the skills", retriever=ragRetreiver, llm=llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

24
Retrieving documents for query: 'What is the skills'
Top K: 3, Score threshold: 0.1
generating embeddings for 1 texts...


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.23it/s]

Generated embeddings with shape: (1, 384)
[1.5833324193954468, 1.5833324193954468, 1.5833324193954468]
Retrieved 3 documents (after filtering)


Answer: The skills of Sanskar Vishwakarma include:

1. Core Concepts: Data Structures, OOP, DBMS, API Integration.
2. Programming Languages: Java, JavaScript, TypeScript, SQL.
3. Libraries and Frameworks: React.js, Next.js, Node.js, Express.js, Spring Boot, REST APIs, Tailwind CSS.
4. Databases: MongoDB, Redis, MySQL.
5. DevOps & Tools: Docker, GitHub, Postman, SoapUI, JIRA, VS Code, IntelliJ IDEA, MS Office.
6. AI Tools: Cursor, Antigravity, Claude, Codex CLI, Gemini CLI, GitHub Copilot.
Sources: [{'source': 'sanskar_resume.pdf', 'page': 0, 'score': 0.38709691114162564, 'preview': 'Sanskar Vishwakarma\n+91 93215 97049 — sanskarv2004@gmail.com — Linkedin — Github\nEducation\nBachelor of Engineering (Computer Engineering)Jun 2021 – Jun 2025\nThakur College of Engineering and Technology9.48 CGPA\nHigher Secondary Certificate (HSC - PCM)Jun 2021\nNirmala College of Science and Commerce9...'}, {'source': 'sanskar_resume.pdf', 'page': 0, 'score': 0.38709691114162564, 'preview': 'Sanskar Vis